#### Goal
- To have a mapping for different sources to compare with GCDS system

#### author
- Abhishek.R.Jaiswal@rabobank.com
- Prajit.Tatari@rabobank.com

##### flow of logic
- Take GCDS data

##### Version & Changes

|     Developer |Date	   | PBI/Bug No |	Changes done |
|----------|----------|----------|----------|
| Abhishek	 |8-July-2025  |12757497    | NLSVF data commented and not capturing. 
| Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load
| Abhishek	 |10-Dec-2025  |14372938    | Extract RANZ data from GRAM 
| Mahalakshmi V | 11-Feb-2025 |15058773 | Included MDM RANZ Source
| Rhea G | 13-Apr-2026 |15580816 | RANZ in RDM v3
| Mahalakshmi V | 20-May-2026 | 16071492 | Included RANZ V4



#####Read Files from GDP

In [0]:
import os
import re
import pandas as pd
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
environment=os.environ['ENV']
radar_datamodel_version_number=3
service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")
NLS_ReadStorage = os.environ['GDP_NA_STORAGE_NAME']
AU_GDP_Defined_Storage_Account=os.environ['AU_GDP_Defined_Storage_Account']

In [0]:
from RadarUtils import *

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_dataobject='Party_SystemIdentifier'

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(GIC_ReadStorage)
authenticate_storage_account(NLS_ReadStorage)
authenticate_storage_account(AU_GDP_Defined_Storage_Account)

In [0]:
# print list of strings for loading spark dfs from GDP
gcds_df = pd.DataFrame({'definedDatasetname':[
'client_Client',
'client_KeyStoreKey',
'client_PartyRole',
'client_PartytoPartyRelationship'
]})

# Create TempView for each loading table
for index, row in gcds_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GCDS', Dataobject=row.definedDatasetname,Load_Date=Load_Date)

In [0]:
# print list of strings for loading spark dfs from GDP
load_df =[
'party_case_client_details',
'party_AllPartyDetails'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService',Load_Date=Load_Date)

In [0]:
# List of dataobjects from GDP
load_df = [
'Party_RiskModelInstanceQuestionAnswers'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='RISKMODEL',Load_Date=Load_Date)

In [0]:
# List of datasets from GDP
load_df =[
'Legacy2_case_client_details'
]
# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='Legacy2', Dataobject=Object,path_prefix='Legacy2',Load_Date=Load_Date)

In [0]:
import pandas as pd # print list of strings for loading spark dfs from GDP
load_df = pd.DataFrame({'GDPname':[
'pessoa'
]})
#gic_load_dts
# Create TempView for each loading table
for index, row in load_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GIC', Dataobject=row.GDPname,Load_Date=Load_Date)

In [0]:
siebel_df = pd.DataFrame({'definedDatasetname':[
'cdf_ggm_org_hist'
]})

for index, row in siebel_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='Siebel', Dataobject=row.definedDatasetname,Load_Date=Load_Date)

In [0]:
load_ranz_v4_rdm_tables(['c_b_party_xref'], Load_Date)

#####GCDS - Other Sources Data Prepration

In [0]:
%sql
Create or replace temporary view Gcob_CaseClientDetails As
select * 
,case when ClientType = 'Legal Entity' then 'Legal Entity' 
      when ClientType in ('Natural Person','Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person' 
      else null
end as Party_type
,Case
      when ClientType = 'Legal Entity' then concat('LEC_', GcobId)
      when ClientType in (
        'Natural Person',
        'Natural Person acting in a Professional Capacity (NPPC)'
      ) then concat('NP_NPPC_', GcobId)
    End as UniquePartyId
from party_case_client_details
where CaseStatusName <> 'Cancelled'
-- and IsLatestApprovedVersionofclient = 'True'

In [0]:
%sql
Create or replace temporary view Gcob_AllPartyDetails_Row As
select p.UniquePartyId
,p.gcdsid
,p.gcobid
,case when p.ClientType = 'LegalEntityClient' then 'Legal Entity' 
      when p.ClientType in ('NaturalPersonClient','Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person' 
      when p.ClientType = 'RelatedLegalEntity' then 'Related Legal Entity' 
      when p.ClientType = 'RelatedNaturalPerson' then 'Related Natural Person' 
      else null
end as Party_type
,ROW_NUMBER() OVER (PARTITION BY p.gcobid,p.ClientType ORDER BY CASE WHEN p.ClientLifeCycleStatus='Client' THEN 1 ELSE 2 END ASC, c.CaseCompletedDate DESC) AS ROWNUM
from party_AllPartyDetails p
Inner join Gcob_CaseClientDetails c on p.UniquePartyId = c.UniqueGcobId
--on CASE WHEN p.ClientType='LegalEntityClient' THEN concat('LEC_',p.Id) WHEN p.ClientType IN ('NaturalPersonClient','Natural Person acting in a Professional Capacity (NPPC)') THEN concat('NP_NPPC_',p.Id) end = c.SourceClient
where Status = 'Live'
--and p.CaseStatusName <> 'Cancelled'
-- and p.IsLatestApprovedVersionofclient = 'True'

In [0]:
%sql
Create or replace temporary view Gcob_AllParty As
With details as 
(
select p.* 
,case when p.ClientType = 'LegalEntityClient' then 'Legal Entity' 
      when p.ClientType in ('NaturalPersonClient','Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person' 
      when p.ClientType = 'RelatedLegalEntity' then 'Related Legal Entity' 
      when p.ClientType = 'RelatedNaturalPerson' then 'Related Natural Person' 
      else null
end as Party_type
,c.SalesforceClientID_nCino
,c.gcobid as ncino_gcobid
,case when p.CaseStatusName is null then 'RelatedParty' else c.CaseStatusName end as StatusName
from party_AllPartyDetails p
LEFT JOIN Gcob_CaseClientDetails c on p.UniquePartyId = c.UniqueGcobId
--on CASE WHEN p.ClientType='LegalEntityClient' THEN concat('LEC_',p.Id) WHEN p.ClientType IN ('NaturalPersonClient','Natural Person acting in a Professional Capacity (NPPC)') THEN concat('NP_NPPC_',p.Id) end = c.SourceClient and p.ClientType in ('NaturalPersonClient','Natural Person acting in a Professional Capacity (NPPC)')
where Status = 'Live'
-- and (p.IsLatestApprovedVersionofclient = 'True' or p.IsLatestApprovedVersionofclient is null)
)
select distinct
*
from details
where StatusName <> 'Cancelled'

In [0]:
%sql
Create or replace temporary view GCDS_Clients As
select distinct
k.KeyStore_value as identifier,k.KeyStore_type,k.status AS status,c.*, pr.Party_role, pr.Life_cycle_status
,rel.`Relationship-Type` as RelationshipType,rel.`Relationship-Value` as RelationshipValue_GCID
from client_KeyStoreKey k
inner join client_Client c on c.GCID = k.GCID
left join client_PartyRole pr on k.GCID = pr.GCID
left join client_PartytoPartyRelationship rel on k.GCID = rel.GCID

In [0]:
%sql
Create or replace temporary view Gcob_AllPartyDetails As
select * from Gcob_AllPartyDetails_Row

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW NonSFDCID_Legacy2_Client AS
select * from Legacy2_case_client_details where isclient = 1 and GcobCaseId is null and Value is null and GcobId like 'RA:%';

CREATE OR REPLACE TEMPORARY VIEW Legacy2_client AS
select * from Legacy2_case_client_details
where ClientId not in (select ClientId from NonSFDCID_Legacy2_Client)
;

CREATE OR REPLACE TEMPORARY VIEW Legacy2 AS
select distinct 
clientId,SalesforceClientID_nCino
,CASE
  WHEN IsClient = 'true' and  ClientTypeId = 1 THEN concat('LE_', GcobId)
  WHEN IsClient = 'true' and ClientTypeId in (2,3) THEN concat('NP_NPPC_', GcobId)
  WHEN IsClient = 'false' and ClientTypeId = 1 THEN concat('RLE_', GcobId)
  WHEN IsClient = 'false' and ClientTypeId in (2,3) THEN concat('RNP_', GcobId)
END as Legacy2_Identifier
,case when ClientType = 'Legal Entity' then 'Legal Entity' 
      when ClientType in ('Natural Person','Natural Person acting in a Professional Capacity (NPPC)') then 'Natural Person' 
      when ClientType = 'Related Legal Entity' then 'Related Legal Entity' 
      when ClientType = 'Related Natural Person' then 'Related Natural Person' 
      else null
end as Party_type
from Legacy2_client 
where ClientTypeId in (1,2,3)
-- and IsLatestApprovedVersionofclient = 'True'

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW SIEBEL_ClientDetails AS
select distinct
s.rel_id as siebelId
FROM cdf_ggm_org_hist AS s
WHERE       
  s.Edl_valid_to_dts like '9999-12-31%'
  AND s.bnk_code IN ('3000', '3400', '3508')
  AND del_f = 'N'
  AND rel_st_tp_ggm_code = 'C'

In [0]:
%sql
Create or replace temporary view GIC_Clients As
select distinct
SEQ_PESSOA 
,COD_INSTITUCIONAL
,CASE WHEN SEQ_TIPO_PESSOA = '2' THEN 'Natural Person' else 'Legal Entity' END AS Party_type
from pessoa

In [0]:
# %sql
# CREATE OR REPLACE TEMPORARY VIEW GRAM_RANZ AS
# select distinct 
# split_part(SourceSystemReferenceId, '_', 1) as CaseID
# ,split_part(SourceSystemReferenceId, '_', 2) as ClientId
# ,SourceSystemReferenceId
# from Party_RiskModelInstanceQuestionAnswers 
# where 
# SourceSystemName = 'RANZ - PegaCdd'
# --and len(SourceSystemReferenceId) >= 15
# and split_part(SourceSystemReferenceId, '_', 2) <> ''

## GCDS Data

In [0]:
%sql
Create or replace temporary view GCDS_Party AS --GCDS_Local_Party As
select distinct
CONCAT('GCDS_',gcid) as PartyIdentifier
,'GCOB' as Application 
,gcob.UniquePartyId as LocalSystemIdentifier
from GCDS_Clients gcds
INNER JOIN Gcob_AllPartyDetails gcob on gcds.identifier = gcob.GcobId
        --and gcds.GCID = gcob.GCDSID
        and gcob.Party_type = 'Legal Entity'
        --and gcds.party_role = 'Customer'
        and gcds.KeyStore_type = 'GCOBID'
where
--gcds.party_role = 'Customer' and 
gcds.Party_type <> 'Natural Person'

UNION

select distinct
CONCAT('GCDS_',gcid) as PartyIdentifier
,'GCOB' as Application
,c.UniquePartyId as LocalSystemIdentifier
from GCDS_Clients gcds
INNER JOIN Gcob_CaseClientDetails c on c.SalesforceClientID_nCino = gcds.identifier 
        and gcds.KeyStore_type = 'NCINOID' 
where
--gcds.party_role = 'Customer' and
c.Party_type = 'Natural Person'

UNION

select distinct
CONCAT('GCDS_',gcid) as PartyIdentifier 
,'LEGACY2' as Application
,l2.Legacy2_Identifier as LocalSystemIdentifier
from GCDS_Clients gcds
INNER JOIN Legacy2 l2 on l2.SalesforceClientID_nCino = gcds.identifier 
        and l2.Party_type = gcds.Party_type
        and gcds.KeyStore_type = 'NCINOID'
--where
--gcds.party_role = 'Customer'

UNION

select distinct
CONCAT('GCDS_',gcid) as PartyIdentifier
,'GIC' as Application
,gic.COD_INSTITUCIONAL as LocalSystemIdentifier
from GCDS_Clients gcds
INNER JOIN GIC_Clients gic on gcds.identifier = gic.COD_INSTITUCIONAL
        and gcds.KeyStore_type = 'GIC'
--where
--gcds.party_role = 'Customer'

UNION

select distinct
CONCAT('GCDS_',gcid) as PartyIdentifier
,'SIEBEL' as Application
,s.SiebelId as LocalSystemIdentifier 
from GCDS_Clients gcds
INNER JOIN SIEBEL_ClientDetails s on gcds.identifier = s.SiebelId

UNION

select distinct
CONCAT('GCDS_',a.gcid) as PartyIdentifier
,'GCDS' as Application
,a.gcid as LocalSystemIdentifier 
from GCDS_Clients a

union

select distinct
CONCAT('GCDS_',gcid) as PartyIdentifier
,'RANZ' as Application
,party_ranz.SRC_PARTY_ID as LocalSystemIdentifier 
from GCDS_Clients gcds
INNER JOIN c_b_party party_ranz on gcds.identifier = party_ranz.SRC_PARTY_ID and gcds.KeyStore_type = 'CB RANZ'

## Non GCDS Data

In [0]:
%sql
Create or replace temporary view GCOB_NonGCDS As
with GCOB as(
select distinct
CONCAT('GCOB_',t1.UniquePartyId) as PartyIdentifier
,'GCOB' as Application
,t1.UniquePartyId as LocalSystemIdentifier
,t1.Party_Type
,t1.caseid
,t1.InternalWatchlist
from 
Gcob_AllParty t1
)

select distinct
PartyIdentifier
,Application
,LocalSystemIdentifier
,ROW_NUMBER() OVER (PARTITION BY PartyIdentifier,Party_Type ORDER BY caseid desc,InternalWatchlist desc) AS ROWNUM
from GCOB 

In [0]:
%sql
Create or replace temporary view NonGCDS_Party As
with NonGCDS as 
(
select * from GCOB_NonGCDS

union

select distinct
CONCAT('GIC_',t1.COD_INSTITUCIONAL) as PartyIdentifier
,'GIC' as Application
,t1.COD_INSTITUCIONAL as LocalSystemIdentifier
,ROW_NUMBER() OVER (PARTITION BY t1.COD_INSTITUCIONAL,t1.Party_Type ORDER BY t1.COD_INSTITUCIONAL DESC) AS ROWNUM
From GIC_Clients t1

union

select distinct
CONCAT('LEGACY2_',t1.Legacy2_Identifier) as PartyIdentifier
,'LEGACY2' as Application
,t1.Legacy2_Identifier AS LocalSystemIdentifier
,ROW_NUMBER() OVER (PARTITION BY t1.Legacy2_Identifier ORDER BY t1.clientid DESC) AS ROWNUM
from Legacy2 as t1

union

select distinct
CONCAT('SIEBEL_',t1.SiebelId) as PartyIdentifier
,'SIEBEL' as Application
,t1.SiebelId AS LocalSystemIdentifier
,ROW_NUMBER() OVER (PARTITION BY t1.SiebelId ORDER BY t1.SiebelId DESC) AS ROWNUM
from SIEBEL_ClientDetails as t1

union

select distinct
CONCAT('RANZ_',SRC_PARTY_ID) as PartyIdentifier
,'RANZ' as Application
,SRC_PARTY_ID as LocalSystemIdentifier
,ROW_NUMBER() OVER (PARTITION BY SRC_PARTY_ID ORDER BY SRC_PARTY_ID DESC) AS ROWNUM 
from c_b_party

)
select  distinct
t1.PartyIdentifier
,t1.Application
,t1.LocalSystemIdentifier
from NonGCDS as t1 where t1.ROWNUM = 1
/*
union

select distinct 
CONCAT('NLSVF_',CIFNUMBER) as PartyIdentifier
,'NLSVF' as Application
,CIFNUMBER AS LocalSystemIdentifier
From nls_dbo_cif
*/

In [0]:
%sql
Create or replace temporary view NonGCDS_UniqueParty As
select * 
from NonGCDS_Party a
left anti join GCDS_Party b
on concat(a.Application,'_',a.LocalSystemIdentifier) = concat(b.Application,'_',b.LocalSystemIdentifier)

##Final Data

In [0]:
%sql
Create or replace temporary view SystemIdentifier_final As
select * from GCDS_Party
union
select * from NonGCDS_UniqueParty

In [0]:
df_SystemIdentifier = spark.table('SystemIdentifier_final')

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_SystemIdentifier, party_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_SystemIdentifier, party_dataobject, radar_datamodel_version_number, environment)